# Multi-Scale Feature Pyramid CNN for Brain Tumor Classification

**Paper:** *Multi-Scale Feature Pyramid CNN for Brain Tumor Classification: Capturing Tumors at Every Size*

**Core idea:** Small meningiomas and large glioblastomas need different receptive fields. A shared EfficientNet-B0 encoder processes the same MRI at 4 scales (224, 112, 56, 28) and fuses them via learned weights.

**Contributions:**
1. Novel 4-scale FPN architecture for brain tumor MRI
2. Scale-tumor size correlation analysis
3. Scale-wise Grad-CAM explainability
4. Benchmark vs single-scale EfficientNet-B0
5. Lightweight learned fusion module

---

## 1. Imports & Config

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR    = '../data'
OUTPUT_DIR  = 'outputs'
MODEL_PATH  = 'outputs/multiscale_fpn_best.pth'
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
SCALES      = (224, 112, 56, 28)
EPOCHS      = 30
BATCH_SIZE  = 32
LR          = 1e-4
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')
print(f'Data    : {DATA_DIR}')

## 2. Dataset

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, split='Training', transform=None):
        self.transform = transform
        self.samples, self.labels = [], []
        root = Path(root_dir) / split
        for idx, cls in enumerate(CLASS_NAMES):
            for ext in ('*.jpg', '*.jpeg', '*.png'):
                for p in (root / cls).glob(ext):
                    self.samples.append(p)
                    self.labels.append(idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = Image.open(self.samples[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

    def get_sampler(self):
        counts  = np.bincount(self.labels)
        weights = 1.0 / counts[self.labels]
        return WeightedRandomSampler(weights, len(weights))


def get_transforms(img_size=224, train=True):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.RandomErasing(p=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def get_dataloaders(data_dir=DATA_DIR, batch_size=BATCH_SIZE):
    train_ds = BrainTumorDataset(data_dir, 'Training', get_transforms(train=True))
    test_ds  = BrainTumorDataset(data_dir, 'Testing',  get_transforms(train=False))
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=train_ds.get_sampler(),
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    return train_loader, test_loader, train_ds, test_ds


train_loader, test_loader, train_ds, test_ds = get_dataloaders()
print(f'Train samples : {len(train_ds)}')
print(f'Test  samples : {len(test_ds)}')

### 2.1 Class Distribution

In [ ]:
train_counts = np.bincount(train_ds.labels)
test_counts  = np.bincount(test_ds.labels)

print(f"{'Class':<15} {'Train':>8} {'Test':>8}")
print('-' * 35)
for cls, tr, te in zip(CLASS_NAMES, train_counts, test_counts):
    print(f'{cls:<15} {tr:>8} {te:>8}')
print(f"{'TOTAL':<15} {sum(train_counts):>8} {sum(test_counts):>8}")

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, counts, title in zip(axes, [train_counts, test_counts], ['Training', 'Testing']):
    bars = ax.bar(CLASS_NAMES, counts, color=colors)
    ax.set_title(f'{title} Set Distribution', fontsize=12)
    ax.set_ylabel('Count')
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(c), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Sample Images Per Class

In [ ]:
shown = {}
for path, label in zip(train_ds.samples, train_ds.labels):
    if label not in shown:
        shown[label] = path
    if len(shown) == 4:
        break

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle('Sample MRI — One Per Class', fontsize=13)
for idx, (label, path) in enumerate(sorted(shown.items())):
    img = Image.open(path).convert('RGB').resize((224, 224))
    axes[idx].imshow(img)
    axes[idx].set_title(CLASS_NAMES[label], fontsize=11)
    axes[idx].axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Multi-Scale Preview — Same MRI at 4 Resolutions

In [ ]:
sample_path = train_ds.samples[0]
orig = Image.open(sample_path).convert('RGB')

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle(f'Same MRI at 4 Scales — {CLASS_NAMES[train_ds.labels[0]]}', fontsize=13)
for ax, scale in zip(axes, SCALES):
    img = orig.resize((scale, scale))
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{scale}×{scale}', fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/multiscale_preview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Model Architecture

In [ ]:
class ScaleEncoder(nn.Module):
    """Shared EfficientNet-B0 backbone — processes one scale at a time."""
    def __init__(self):
        super().__init__()
        backbone    = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.pool     = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        return self.pool(self.features(x)).flatten(1)  # (B, 1280)


class LearnedFusion(nn.Module):
    """Weighted sum of scale features with a learnable per-scale weight."""
    def __init__(self, num_scales=4, feature_dim=1280):
        super().__init__()
        self.scale_weights = nn.Parameter(torch.ones(num_scales) / num_scales)
        self.fc = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.SiLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
        )

    def forward(self, features):
        weights = F.softmax(self.scale_weights, dim=0)
        fused   = sum(w * f for w, f in zip(weights, features))
        return self.fc(fused), weights


class MultiScaleFPN(nn.Module):
    """
    Multi-Scale Feature Pyramid Network for brain tumor classification.
    Processes MRI at SCALES resolutions with a shared EfficientNet-B0 encoder,
    fuses via learned weighted sum, then classifies.
    """
    SCALES = (224, 112, 56, 28)

    def __init__(self, num_classes=4):
        super().__init__()
        self.encoder    = ScaleEncoder()
        self.fusion     = LearnedFusion(num_scales=len(self.SCALES))
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        scale_features = []
        for scale in self.SCALES:
            xs = F.interpolate(x, size=(scale, scale), mode='bilinear', align_corners=False) \
                 if scale != x.shape[-1] else x
            scale_features.append(self.encoder(xs))
        fused, scale_weights = self.fusion(scale_features)
        return self.classifier(fused), scale_weights


# ── Parameter count ───────────────────────────────────────────────────────────
model = MultiScaleFPN(num_classes=4)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Scales            : {model.SCALES}')
print(f'Total parameters  : {total:,}')
print(f'Trainable params  : {trainable:,}')

### 3.1 Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')

box_style = dict(boxstyle='round,pad=0.5', facecolor='#3498db', alpha=0.8, edgecolor='white')
enc_style = dict(boxstyle='round,pad=0.5', facecolor='#e74c3c', alpha=0.8, edgecolor='white')
fus_style = dict(boxstyle='round,pad=0.5', facecolor='#2ecc71', alpha=0.8, edgecolor='white')
cls_style = dict(boxstyle='round,pad=0.5', facecolor='#f39c12', alpha=0.8, edgecolor='white')

ax.text(0.05, 0.5, 'MRI Input\n224×224', ha='center', va='center',
        transform=ax.transAxes, fontsize=10, color='white', bbox=box_style)

scale_labels = ['224×224', '112×112', '56×56', '28×28']
ys = [0.8, 0.6, 0.4, 0.2]
for lbl, y in zip(scale_labels, ys):
    ax.text(0.28, y, f'Scale\n{lbl}', ha='center', va='center',
            transform=ax.transAxes, fontsize=9, color='white', bbox=enc_style)
    ax.text(0.50, y, 'EfficientNet-B0\n(shared)', ha='center', va='center',
            transform=ax.transAxes, fontsize=9, color='white', bbox=enc_style)
    ax.annotate('', xy=(0.42, y), xytext=(0.34, y),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    ax.annotate('', xy=(0.64, 0.5), xytext=(0.57, y),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))

ax.annotate('', xy=(0.22, 0.5), xytext=(0.10, 0.5),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.text(0.70, 0.5, 'Learned\nFusion', ha='center', va='center',
        transform=ax.transAxes, fontsize=10, color='white', bbox=fus_style)
ax.text(0.88, 0.5, '4-Class\nOutput', ha='center', va='center',
        transform=ax.transAxes, fontsize=10, color='white', bbox=cls_style)

ax.annotate('', xy=(0.82, 0.5), xytext=(0.76, 0.5),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.set_title('Multi-Scale FPN Architecture', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/architecture.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Training

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits, _ = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits, _    = model(imgs)
        loss = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        preds       = logits.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += len(labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


# ── Init ──────────────────────────────────────────────────────────────────────
model     = MultiScaleFPN(num_classes=4).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history   = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc  = 0.0

print(f'Training for {EPOCHS} epochs on {DEVICE}...\n')

for epoch in tqdm(range(1, EPOCHS + 1), desc='Epochs'):
    tr_loss, tr_acc               = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, preds, lbs = eval_epoch(model, test_loader, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'class_names': CLASS_NAMES,
            'scales': model.SCALES,
        }, MODEL_PATH)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}/{EPOCHS} | '
              f'Train Loss {tr_loss:.4f}  Acc {tr_acc:.4f} | '
              f'Val Loss {val_loss:.4f}  Acc {val_acc:.4f}')

with open(f'{OUTPUT_DIR}/history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nBest Val Accuracy : {best_acc:.4f}')
print(f'Model saved to    : {MODEL_PATH}')

## 5. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train', linewidth=2, color='#e74c3c')
ax1.plot(history['val_loss'],   label='Val',   linewidth=2, color='#3498db')
ax1.set_title('Cross-Entropy Loss', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='Train', linewidth=2, color='#e74c3c')
ax2.plot(history['val_acc'],   label='Val',   linewidth=2, color='#3498db')
ax2.set_title('Accuracy', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Multi-Scale FPN — Training Curves', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best Val Accuracy : {max(history["val_acc"]):.4f}')

## 6. Evaluation

### 6.1 Classification Report

In [ ]:
# Load best model
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded best model (val_acc={ckpt["val_acc"]:.4f} @ epoch {ckpt["epoch"]})')

all_preds, all_labels, all_probs, all_weights = [], [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        logits, scale_w = model(imgs)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.append(probs)
        all_weights.append(scale_w.cpu().numpy())

all_preds   = np.array(all_preds)
all_labels  = np.array(all_labels)
all_probs   = np.vstack(all_probs)
all_weights = np.vstack(all_weights)

print('\nClassification Report')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

### 6.2 Macro-AUC

In [ ]:
labels_bin = label_binarize(all_labels, classes=list(range(4)))
auc = roc_auc_score(labels_bin, all_probs, multi_class='ovr', average='macro')
print(f'Macro-AUC : {auc:.4f}')

### 6.3 Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — Multi-Scale FPN', fontsize=13)
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.4 Scale Importance Per Tumor Type

In [ ]:
print('Scale Importance Per Tumor Type')
print(f'{"Class":<14}', end='')
for s in SCALES:
    print(f'  {s}×{s}', end='')
print()
print('-' * 62)

means = []
for idx, cls in enumerate(CLASS_NAMES):
    mask   = all_labels == idx
    mean_w = all_weights[mask].mean(axis=0)
    means.append(mean_w)
    print(f'{cls:<14}', end='')
    for w in mean_w:
        print(f'  {w:.4f}', end='')
    print()

means = np.array(means)
x     = np.arange(len(CLASS_NAMES))
width = 0.18
colors_scale = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (scale, col) in enumerate(zip(SCALES, colors_scale)):
    ax.bar(x + i * width, means[:, i], width, label=f'{scale}×{scale}', color=col, alpha=0.85)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylabel('Mean Scale Weight')
ax.set_title('Scale Importance Per Tumor Type\n(higher = model relies more on this resolution)', fontsize=12)
ax.legend(title='Scale')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/scale_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Scale-Wise Grad-CAM

In [ ]:
class ScaleWiseGradCAM:
    """Grad-CAM applied independently at each FPN scale."""

    def __init__(self, model):
        self.model       = model
        self._activations = None
        self._gradients   = None
        model.encoder.features[-1].register_forward_hook(self._save_act)
        model.encoder.features[-1].register_full_backward_hook(self._save_grad)

    def _save_act(self, _, __, output):
        self._activations = output.detach()

    def _save_grad(self, _, __, grad_output):
        self._gradients = grad_output[0].detach()

    def _cam_at_scale(self, x, class_idx):
        self.model.zero_grad()
        feat   = self.model.encoder.features(x)
        pooled = self.model.encoder.pool(feat).flatten(1)
        pooled[0, class_idx].backward(retain_graph=True)
        weights = self._gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self._activations).sum(dim=1).squeeze())
        cam = cam.cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def visualize(self, image_path, save_path=None):
        self.model.eval()
        transform = get_transforms(224, train=False)
        img    = Image.open(image_path).convert('RGB')
        tensor = transform(img).unsqueeze(0).to(DEVICE)

        logits, scale_weights = self.model(tensor)
        pred_idx = logits.argmax(1).item()
        weights  = scale_weights.detach().cpu().numpy()
        orig_np  = np.array(img.resize((224, 224))) / 255.0

        cams = {}
        for scale in self.model.SCALES:
            xs = F.interpolate(tensor, size=(scale, scale), mode='bilinear', align_corners=False) \
                 if scale != tensor.shape[-1] else tensor
            cams[scale] = self._cam_at_scale(xs, pred_idx)

        fig, axes = plt.subplots(2, len(self.model.SCALES), figsize=(16, 7))
        fig.suptitle(f'Scale-Wise Grad-CAM  |  Predicted: {CLASS_NAMES[pred_idx]}', fontsize=13)

        for i, scale in enumerate(self.model.SCALES):
            cam_r = np.array(
                Image.fromarray((cams[scale] * 255).astype(np.uint8)).resize((224, 224))
            ) / 255.0
            jet_rgb = plt.cm.jet(cam_r)[:, :, :3]

            axes[0, i].imshow(cam_r, cmap='jet')
            axes[0, i].set_title(f'{scale}×{scale}\nWeight: {weights[i]:.3f}', fontsize=10)
            axes[0, i].axis('off')

            overlay = np.clip(0.5 * orig_np + 0.5 * jet_rgb, 0, 1)
            axes[1, i].imshow(overlay)
            axes[1, i].set_title('Overlay', fontsize=10)
            axes[1, i].axis('off')

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        return pred_idx, weights


print('ScaleWiseGradCAM ready.')

In [ ]:
# Run Grad-CAM on one sample per class
cam_engine = ScaleWiseGradCAM(model)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    # Pick first test image of this class
    img_path = next(
        str(p) for p, l in zip(test_ds.samples, test_ds.labels) if l == cls_idx
    )
    print(f'\n--- {cls_name.upper()} ---')
    pred_idx, weights = cam_engine.visualize(
        img_path,
        save_path=f'{OUTPUT_DIR}/gradcam_{cls_name}.png'
    )
    print(f'Predicted : {CLASS_NAMES[pred_idx]}')
    print(f'Scale weights : {dict(zip(SCALES, weights.round(3)))}')

## 8. Ablation — Multi-Scale FPN vs Single-Scale EfficientNet-B0

In [ ]:
class SingleScaleBaseline(nn.Module):
    """Plain EfficientNet-B0 — the single-scale baseline to beat."""
    def __init__(self, num_classes=4):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_feat  = backbone.classifier[1].in_features
        backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_feat, 512),
            nn.SiLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )
        self.model = backbone

    def forward(self, x):
        return self.model(x)


def run_epoch_generic(model, loader, optimizer, criterion, is_fpn, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(imgs)[0] if is_fpn else model(imgs)
            loss   = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)
    return total_loss / total, correct / total


def train_and_eval(model, is_fpn, label):
    crit  = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt   = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = CosineAnnealingLR(opt, T_max=EPOCHS)
    best  = 0.0
    for epoch in tqdm(range(EPOCHS), desc=f'  {label}'):
        run_epoch_generic(model, train_loader, opt, crit, is_fpn, train=True)
        _, val_acc = run_epoch_generic(model, test_loader, None, crit, is_fpn, train=False)
        sched.step()
        best = max(best, val_acc)

    # Final detailed eval
    model.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs   = imgs.to(DEVICE)
            logits = model(imgs)[0] if is_fpn else model(imgs)
            preds.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(lbls.numpy())

    acc = sum(p == l for p, l in zip(preds, labels_list)) / len(labels_list)
    f1  = f1_score(labels_list, preds, average='macro')
    return acc, f1


ablation_results = {}

print('Training Single-Scale Baseline...')
baseline = SingleScaleBaseline().to(DEVICE)
acc, f1  = train_and_eval(baseline, is_fpn=False, label='Baseline')
ablation_results['SingleScale-EfficientNet-B0'] = {'accuracy': acc, 'macro_f1': f1}
print(f'  Accuracy {acc:.4f}  Macro-F1 {f1:.4f}\n')

print('Training Multi-Scale FPN...')
fpn_model = MultiScaleFPN().to(DEVICE)
acc, f1   = train_and_eval(fpn_model, is_fpn=True, label='FPN')
ablation_results['MultiScale-FPN'] = {'accuracy': acc, 'macro_f1': f1}
print(f'  Accuracy {acc:.4f}  Macro-F1 {f1:.4f}')

with open(f'{OUTPUT_DIR}/comparison.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

### 8.1 Comparison Chart

In [ ]:
models_   = list(ablation_results.keys())
accs_     = [ablation_results[m]['accuracy']  for m in models_]
f1_scores = [ablation_results[m]['macro_f1']  for m in models_]
bar_colors = ['#95a5a6', '#e74c3c']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

for ax, vals, title in zip([ax1, ax2], [accs_, f1_scores], ['Accuracy', 'Macro-F1']):
    bars = ax.bar(models_, vals, color=bar_colors, width=0.45)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(max(0, min(vals) - 0.05), 1.0)
    ax.set_xticklabels(models_, rotation=10, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')

gain = (ablation_results['MultiScale-FPN']['accuracy'] -
        ablation_results['SingleScale-EfficientNet-B0']['accuracy']) * 100

plt.suptitle(f'Single-Scale vs Multi-Scale FPN  (FPN gain: +{gain:.2f}%)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Final Results Summary

In [ ]:
fpn_acc  = ablation_results['MultiScale-FPN']['accuracy']
fpn_f1   = ablation_results['MultiScale-FPN']['macro_f1']
base_acc = ablation_results['SingleScale-EfficientNet-B0']['accuracy']
base_f1  = ablation_results['SingleScale-EfficientNet-B0']['macro_f1']

print('=' * 60)
print(f'{"Model":<35} {"Accuracy":>10} {"Macro-F1":>10}')
print('-' * 60)
print(f'{"SingleScale-EfficientNet-B0":<35} {base_acc:>10.4f} {base_f1:>10.4f}')
print(f'{"MultiScale-FPN (ours)":<35} {fpn_acc:>10.4f} {fpn_f1:>10.4f}')
print('=' * 60)
print(f'\nAccuracy gain : +{(fpn_acc - base_acc)*100:.2f}%')
print(f'Macro-F1 gain : +{(fpn_f1  - base_f1 )*100:.2f}%')
print(f'AUC           : {auc:.4f}')
print(f'\nOutputs saved to: {OUTPUT_DIR}/')
print('  - training_curves.png')
print('  - confusion_matrix.png')
print('  - scale_importance.png')
print('  - gradcam_<class>.png  (×4)')
print('  - model_comparison.png')
print('  - multiscale_fpn_best.pth')